# 1.0 Dataset Introduction

The Kepler space telescope was launched by NASA in 2009 to find planets orbiting stars outside our solar system (exoplanets). It found them using the transit method: it continuously measured the brightness of around 150,000 stars and recorded the small, regular dips in light that occur when a planet passes in front of its star. Each repeating dip worth investigating is logged as a Kepler Object of Interest (KOI).

Not every KOI is a planet. Similar dips can be caused by two stars eclipsing each other, by a nearby star contaminating the measurement, or by instrument noise. Each KOI is therefore reviewed and given a disposition: confirmed planet, candidate still awaiting confirmation, or false positive. Doing this by hand is slow, which makes the dataset well suited to machine learning. A model can learn, from the measured properties of the signal and its host star, to separate real planets from false positives.

# 1.1 Dataset Identification

The dataset was retrieved from Kaggle, where it was uploaded under NASA's account as "Kepler Exoplanet Search Results". It is a copy of the Kepler Objects of Interest (KOI) cumulative table, which is published by the NASA Exoplanet Archive (NExScI-Caltech/IPAC, doi:10.26133/NEA4). The table brings together the results of NASA's searches of light curves from the Kepler space telescope. Although the copy was obtained from Kaggle, the data originate from a primary scientific archive and can be checked against it. The dataset can be found at the following URL: 

https://www.kaggle.com/datasets/nasa/kepler-exoplanet-search-results

The dataset contains fifty (50) attributes over 9,564 rows, where each row is one candidate planet. The target is koi_disposition. Four attributes are identifiers and are excluded from modelling, which leaves forty-five (45) features. The attributes are defined as the following types:

- <mark>Categorical (Nominal)</mark>: koi_disposition (CONFIRMED, CANDIDATE, FALSE POSITIVE), koi_pdisposition (CANDIDATE, FALSE POSITIVE), koi_tce_delivname (one of three data releases).
- <mark>Categorical (Binary)</mark>: koi_fpflag_nt, koi_fpflag_ss, koi_fpflag_co, koi_fpflag_ec These are stored as 0/1 flags, each marking a specific reason the signal may be a false positive.
- <mark>Identifiers</mark>: rowid, kepid, kepoi_name, kepler_name These label the object and carry no predictive meaning. kepler_name is only filled for confirmed planets, so 76% of its values are missing.
- <mark>Numerical (Discrete)</mark>: koi_tce_plnt_num The planet's number within its star system, an integer between 1 and 8.
- <mark>Numerical (Continuous)</mark>: koi_score (0 to 1), koi_period, koi_impact, koi_duration, koi_depth, koi_prad, koi_teq, koi_insol, koi_model_snr, koi_steff, koi_slogg, koi_srad, ra, dec and koi_kepmag. Most of these come with an upper (_err1) and lower (_err2) uncertainty, which are also continuous.
- <mark>Time</mark>: koi_time0bk The time of the first observed transit. It is stored as a decimal number of days in Barycentric Kepler Julian Date (BKJD) rather than as a date string, for example 170.53875.
koi_teq_err1 and koi_teq_err2 are entirely empty and koi_score is missing for 16% of rows. koi_score, koi_pdisposition and the four false-positive flags are closely linked to the target, so they should be excluded when classifying koi_disposition. Table 2 summarises the attributes.

| Attribute(s) | Data type | Variable type | Description |
|---|---|---|---|
| rowid, kepid, kepoi_name, kepler_name | Integer / String | Identifier | Row, star and planet identifiers |
| koi_disposition | String | Categorical (nominal) | Target: archive disposition of the candidate |
| koi_pdisposition | String | Categorical (nominal) | Disposition using Kepler data alone |
| koi_tce_delivname | String | Categorical (nominal) | Data release the detection came from |
| koi_fpflag_nt / ss / co / ec | Integer | Categorical (binary) | False-positive flags: not transit-like, stellar eclipse, centroid offset, ephemeris match |
| koi_tce_plnt_num | Float | Numerical (discrete) | Planet number within the system |
| koi_time0bk | Float | Time | Transit epoch (BKJD days) |
| koi_score | Float | Numerical (continuous) | Confidence in the disposition, 0 to 1 |
| koi_period, koi_duration, koi_depth, koi_impact | Float | Numerical (continuous) | Orbital period (days), transit duration (hours), transit depth (ppm), impact parameter |
| koi_prad, koi_teq, koi_insol | Float | Numerical (continuous) | Planet radius (Earth radii), equilibrium temperature (K), insolation flux |
| koi_model_snr | Float | Numerical (continuous) | Transit signal-to-noise ratio |
| koi_steff, koi_slogg, koi_srad, koi_kepmag | Float | Numerical (continuous) | Star temperature (K), surface gravity, radius (Solar radii), Kepler magnitude |
| ra, dec | Float | Numerical (continuous) | Sky coordinates (degrees) |
| *_err1, *_err2 | Float | Numerical (continuous) | Upper and lower uncertainty of the matching attribute |

## 1.2. Supervised Learning Task Identification

**Financial Regression**

**Kepler Objects of Interest**

Herein, the objective of the supervised learning task is to use classification techniques to accurately categorise each Kepler Object of Interest by its disposition (koi_disposition). This can also be determined from the following:

- The target variable, koi_disposition, is categorical (nominal), with three possible classes: CONFIRMED, CANDIDATE and FALSE POSITIVE.
- The features describe measured properties of the transit signal and its host star, which are used to decide whether a signal belongs to a real planet or a false positive.
- The classes are imbalanced, with false positives making up over half of all rows (5,023 of 9,564), which should be taken into account when training and evaluating a model.

A regression task can also be performed on the continuous attributes, such as predicting a planet's radius (koi_prad) from its transit depth and host-star properties. However, classification is the dataset's primary purpose.

## 2 Exploratory Data Analysis

With the identification of the dataset, and the supervised learning task being defined as Kepler Objects of Interest and classification respectively, exploratory data analysis (EDA) can be undertaken to facilitate a deeper understanding of the structure of the data, as well as allowing for identification of issues with data quality (IBM, 2024). This is particularly important for this dataset, as it contains identifier columns, attributes that are entirely empty (koi_teq_err1 and koi_teq_err2), and attributes such as koi_score and the false-positive flags that are closely linked to the target. Furthermore, by creating visualisations of the dataset, it is easier to identify any patterns, trends, or correlations that separate confirmed planets from false positives. With conclusions drawn from this exploration, suitable machine learning algorithms and models can be identified, and features can be engineered to improve the performance of the chosen model(s). These model(s) can then be evaluated for their efficacy when provided data of a similar format to the original dataset.

Within the following section, questions and assumptions about the dataset will be proposed. Then an explanation of how data will be split into training, validation, and testing sets will be provided; as the classes are imbalanced, the split will be stratified so that each set contains the same proportion of each disposition. This will be followed by EDA, defining the process and results, which will then be measured against the previously proposed questions and assumptions.

### 2.1 Question(s) identification

Research questions should allow the researcher or investigator to set out what they want to know as a hypothesis, regardless of whether the result support the hypothesis or not (Mayo et al., 2013). By providing well posed questions and assumptions the EDA process will be able to support or reject their claims (Lane, 2018). Table 2 below provides several questions related to the dataset and their associated assumptions.

| ID | Question | Assumption |
|---|---|---|
| RQ1 | Does transit depth (koi_depth) differ between confirmed planets and false positives? | A common cause of false positives is two stars eclipsing each other, which blocks far more light than a planet. It is expected that false positives will show much greater transit depths than confirmed planets. |
| RQ2 | Does planet radius (koi_prad) differ between confirmed planets and false positives? | As planet radius is calculated from transit depth, it is expected that false positives will show implausibly large radii (larger than the largest known planets), while confirmed planets will mostly be smaller than Jupiter. |
| RQ3 | Does the impact parameter (koi_impact) differ between confirmed planets and false positives? | The impact parameter describes how close to the centre of the star the object passes. It is expected that false positives will more often show high values (above 1), suggesting a grazing eclipse rather than a planet passing in front of the star. |
| RQ4 | Does orbital period (koi_period) impact disposition? | It is expected that confirmed planets will more often have short orbital periods, as these produce more transits during the mission and are therefore easier to confirm. |
| RQ5 | Does the number of planets in a system (koi_tce_plnt_num) impact disposition? | It is expected that objects in systems with more than one candidate are much more likely to be confirmed planets, as it is unlikely for several false signals to occur around the same star (Lissauer et al., 2012). |
| RQ6 | Do the properties of the host star (koi_steff, koi_srad) impact disposition? | It is expected that host-star temperature and radius alone will have little impact on disposition, as planets and false positives can occur around stars of any type. |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import accuracy_score, recall_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
DATA_PATH = "../datasets/"
kep = pd.read_csv(DATA_PATH + "cumulative.csv")
print(kep.shape)
kep.head()

In [16]:
kep.isnull().sum()

koi_disposition      0
koi_pdisposition     0
koi_fpflag_nt        0
koi_fpflag_ss        0
koi_fpflag_co        0
koi_fpflag_ec        0
koi_period           0
koi_period_err1      0
koi_period_err2      0
koi_time0bk          0
koi_time0bk_err1     0
koi_time0bk_err2     0
koi_impact           0
koi_impact_err1      0
koi_impact_err2      0
koi_duration         0
koi_duration_err1    0
koi_duration_err2    0
koi_depth            0
koi_depth_err1       0
koi_depth_err2       0
koi_prad             0
koi_prad_err1        0
koi_prad_err2        0
koi_teq              0
koi_insol            0
koi_insol_err1       0
koi_insol_err2       0
koi_model_snr        0
koi_tce_plnt_num     0
koi_tce_delivname    0
koi_steff            0
koi_steff_err1       0
koi_steff_err2       0
koi_slogg            0
koi_slogg_err1       0
koi_slogg_err2       0
koi_srad             0
koi_srad_err1        0
koi_srad_err2        0
ra                   0
dec                  0
koi_kepmag           0
dtype: int6

In [15]:
# Remove identifiers and the target-linked score
kep = kep.drop(columns=["rowid", "kepid", "kepoi_name", "kepler_name", "koi_score"])

# Remove columns that are entirely empty
kep = kep.drop(columns=["koi_teq_err1", "koi_teq_err2"])

# Remove the rows that still have gaps
kep = kep.dropna()

### Split the Dataset

In [ ]:
target = "koi_disposition"

leaky = ["koi_pdisposition", "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"]
X = kep.drop([target] + leaky, axis= 1)
y = kep[target]

# Split the dataset into 60% Training data, 40% Temporary dataset with a random state of 42
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40, stratify=y, random_state=42)

# Further Split the temporary dataset into validation data (20%), testing data (20%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

In [18]:
kep.info()

<class 'pandas.DataFrame'>
Index: 8744 entries, 0 to 9563
Data columns (total 43 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   koi_disposition    8744 non-null   str    
 1   koi_pdisposition   8744 non-null   str    
 2   koi_fpflag_nt      8744 non-null   int64  
 3   koi_fpflag_ss      8744 non-null   int64  
 4   koi_fpflag_co      8744 non-null   int64  
 5   koi_fpflag_ec      8744 non-null   int64  
 6   koi_period         8744 non-null   float64
 7   koi_period_err1    8744 non-null   float64
 8   koi_period_err2    8744 non-null   float64
 9   koi_time0bk        8744 non-null   float64
 10  koi_time0bk_err1   8744 non-null   float64
 11  koi_time0bk_err2   8744 non-null   float64
 12  koi_impact         8744 non-null   float64
 13  koi_impact_err1    8744 non-null   float64
 14  koi_impact_err2    8744 non-null   float64
 15  koi_duration       8744 non-null   float64
 16  koi_duration_err1  8744 non-null   float

In [ ]:
unique_values = 